Aquí importamos lo necesario

In [ ]:
import cv2
import numpy as np
import csv
import math
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

**Tarea 1:** Cálculo del valor total de monedas

In [4]:
vid = cv2.VideoCapture(0)

# Diámetros oficiales en milímetros
diametro_monedas = {
    2.00: 25.75,
    1.00: 23.25,
    0.50: 24.25,
    0.20: 22.25,
    0.10: 19.75,
    0.05: 21.25,
    0.02: 18.75,
    0.01: 16.25
}

# Variables globales
moneda_seleccionada = None
diametro_ref = None
escala = None

def clasificar_moneda(diametro_medido, diccionario):
    moneda = min(
        diccionario,
        key=lambda valor: abs(diccionario[valor] - diametro_medido)
    )
    return moneda

def seleccionar_moneda(event, x, y, flags, param):
    global moneda_seleccionada, diametro_ref, escala

    if event == cv2.EVENT_LBUTTONDOWN:
        for c in contornos:
            if cv2.pointPolygonTest(c, (x, y), False) >= 0:
                # Medir la moneda seleccionada mediante el círculo mínimo
                centro, radio = cv2.minEnclosingCircle(c)
                diametro_ref = radio * 2

                # La moneda seleccionada se considera de 1 €
                escala = diametro_monedas[1.00] / diametro_ref
                moneda_seleccionada = (x, y)

                break

cv2.namedWindow("Vid")
cv2.setMouseCallback("Vid", seleccionar_moneda)

while True:
    ret, frame = vid.read()

    if ret:
        # Convertir a escala de grises
        img_gris = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        # Suavizado para reducir ruido
        img_suavizada = cv2.GaussianBlur(
            img_gris, (5, 5), 0
        )

        # Umbral automático de Otsu
        _, img_th = cv2.threshold(
            img_suavizada,
            0,
            255,
            cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU
        )

        # Detectar contornos externos
        contornos, _ = cv2.findContours(
            img_th,
            cv2.RETR_EXTERNAL,
            cv2.CHAIN_APPROX_SIMPLE
        )

        total = 0.0

        # Dibuja todos los contornos mientras no haya una escala
        if escala is None:
            cv2.drawContours(frame, contornos, -1, (0, 255, 0), 2)
        
        else:
            for c in contornos:
                centro, radio = cv2.minEnclosingCircle(c)
                diametro_px = radio * 2

                # Ignorar objetos demasiado pequeños y demasiado grandes
                if diametro_px < diametro_ref * 0.55:
                    continue

                if diametro_px > diametro_ref * 1.15:
                    continue

                # Solo dibuja los contornos de las monedas
                cv2.drawContours(frame, [c], -1, (0, 255, 0), 2)

                # Convertir píxeles a milímetros
                diametro_mm = diametro_px * escala

                # Clasificar por el diámetro oficial más cercano
                valor_moneda = clasificar_moneda(
                    diametro_mm,
                    diametro_monedas
                )

                # Dibujar el valor en el centro de la moneda
                x_centro = int(centro[0])
                y_centro = int(centro[1])

                texto = f"{valor_moneda:.2f} EUR"

                cv2.putText(
                    frame,
                    texto,
                    (x_centro - 45, y_centro),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.6,
                    (0, 0, 255),
                    2,
                    cv2.LINE_AA
                )

                # Sumar el valor clasificado
                total += valor_moneda

            # Mostrar el total
            cv2.putText(
                frame,
                f"TOTAL = {total:.2f} EUR",
                (10, 40),
                cv2.FONT_HERSHEY_SIMPLEX,
                1,
                (0, 0, 255),
                2,
                cv2.LINE_AA
            )

        cv2.imshow("Vid", frame)
        cv2.imshow("Umbral", img_th)

    # ESC para salir
    if cv2.waitKey(20) == 27:
        break

vid.release()
cv2.destroyAllWindows()

**Tarea 2:**: Detección y Clasificación de Microplásticos

In [ ]:
def rellenar_huecos(binaria):
    flood = binaria.copy()
    mascara = np.zeros((binaria.shape[0] + 2, binaria.shape[1] + 2), np.uint8)
    cv2.floodFill(flood, mascara, (0, 0), 255)
    return binaria | cv2.bitwise_not(flood)


def extraer_caracteristicas_completas(contorno, imagen_gris):
    area = cv2.contourArea(contorno)
    if area < 15: return None
        
    perimetro = cv2.arcLength(contorno, True)
    if perimetro == 0: return None
        
    compacidad = (perimetro ** 2) / area
    circularidad = (4.0 * math.pi * area) / (perimetro ** 2)
    

    hull = cv2.convexHull(contorno)
    hull_area = cv2.contourArea(hull)
    solidez = area / hull_area if hull_area > 0 else 0
    perimetro_hull = cv2.arcLength(hull, True)
    rugosidad = perimetro / perimetro_hull if perimetro_hull > 0 else 1.0
    
    x, y, w, h = cv2.boundingRect(contorno)
    extent = area / (w * h) if w * h > 0 else 0
    aspect_ratio = float(min(w, h)) / max(w, h) if max(w, h) > 0 else 0
    
    if len(contorno) >= 5:
        _, (eje_menor, eje_mayor), _ = cv2.fitEllipse(contorno)
        axis_ratio = eje_menor / eje_mayor if eje_mayor > 0 else 0
    else:
        axis_ratio = aspect_ratio
        
    M = cv2.moments(contorno)
    hu = cv2.HuMoments(M).flatten()
    hu_log = [-math.copysign(math.log10(abs(val)), val) if val != 0 else 0 for val in hu[:2]]

    mask = np.zeros(imagen_gris.shape, dtype=np.uint8)
    cv2.drawContours(mask, [contorno], -1, 255, -1)
    mean_val, std_val = cv2.meanStdDev(imagen_gris, mask=mask)
    
    return [compacidad, circularidad, solidez, rugosidad, extent, aspect_ratio, axis_ratio, hu_log[0], hu_log[1], mean_val[0][0], std_val[0][0]]


def segmentar_particulas(img_gris):
    img_blur = cv2.GaussianBlur(img_gris, (5, 5), 0)
    th = cv2.adaptiveThreshold(img_blur, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 25, 3)
    
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    th = cv2.morphologyEx(th, cv2.MORPH_CLOSE, kernel, iterations=2)
    th = rellenar_huecos(th)
    th = cv2.morphologyEx(th, cv2.MORPH_OPEN, kernel, iterations=1)
    
    contornos, _ = cv2.findContours(th, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    return contornos


X_train, y_train = [], []
train_images_info = [('FRA.png', 'FRA'), ('PEL.png', 'PEL'), ('TAR.png', 'TAR')]

plt.figure(figsize=(15, 5))

for idx, (img_file, label) in enumerate(train_images_info):
    img = cv2.imread(img_file, cv2.IMREAD_GRAYSCALE)
    if img is None:
        continue

    contornos = segmentar_particulas(img)
    img_vis = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    contornos_validos = []

    for cnt in contornos:
        feats = extraer_caracteristicas_completas(cnt, img)
        if feats is not None:
            X_train.append(feats)
            y_train.append(label)
            contornos_validos.append(cnt)

    cv2.drawContours(img_vis, contornos_validos, -1, (0, 255, 0), 2)
    cv2.imwrite(f"deteccion_{label}.png", img_vis)

    plt.subplot(1, 3, idx + 1)
    plt.imshow(cv2.cvtColor(img_vis, cv2.COLOR_BGR2RGB))
    plt.title(f"{label} ({len(contornos_validos)})")
    plt.axis("off")

plt.tight_layout()
plt.show()

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

clf = RandomForestClassifier(
    n_estimators=300, 
    max_depth=9, 
    min_samples_leaf=2,
    random_state=42, 
    class_weight={'FRA': 0.9, 'PEL': 3.5, 'TAR': 1.6}
)
clf.fit(X_train_scaled, y_train)


img_test = cv2.imread("MPs_test.jpg", cv2.IMREAD_GRAYSCALE)
csv_file = "MPs_test_bbs.csv"

X_test, y_true = [], []
bbs_test = []
margen = 15 

with open(csv_file, newline="", encoding="utf-8") as file:
    reader = csv.DictReader(file)
    for row in reader:
        etiqueta_real = row["label"]
        x_min, y_min, x_max, y_max = map(int, [row["x_min"], row["y_min"], row["x_max"], row["y_max"]])
        
        y_min_m, y_max_m = max(0, y_min - margen), min(img_test.shape[0], y_max + margen)
        x_min_m, x_max_m = max(0, x_min - margen), min(img_test.shape[1], x_max + margen)
        
        roi = img_test[y_min_m:y_max_m, x_min_m:x_max_m]
        if roi.size == 0: continue
            
        contornos_roi = segmentar_particulas(roi)
        
        if contornos_roi:
            centro_roi_x = roi.shape[1] // 2
            centro_roi_y = roi.shape[0] // 2
            
            mejor_contorno = None
            min_distancia = float('inf')
            
            for c in contornos_roi:
                if cv2.contourArea(c) < 15 or cv2.contourArea(c) > (roi.shape[0]*roi.shape[1]*0.9): 
                    continue
                
                M = cv2.moments(c)
                if M["m00"] != 0:
                    cx, cy = int(M["m10"] / M["m00"]), int(M["m01"] / M["m00"])
                    dist = math.sqrt((cx - centro_roi_x)**2 + (cy - centro_roi_y)**2)
                    if dist < min_distancia:
                        min_distancia = dist
                        mejor_contorno = c
            
            if mejor_contorno is not None:
                feats = extraer_caracteristicas_completas(mejor_contorno, roi)
                if feats is not None:
                    X_test.append(feats)
                    y_true.append(etiqueta_real)
                    bbs_test.append((x_min, y_min, x_max, y_max))

X_test_scaled = scaler.transform(X_test)
y_pred = clf.predict(X_test_scaled)

img_test_color = cv2.imread("MPs_test.jpg", cv2.IMREAD_COLOR)

colores = {
    'FRA': (255, 0, 0),
    'PEL': (0, 255, 0),
    'TAR': (0, 0, 255),
}

for (x_min, y_min, x_max, y_max), pred in zip(bbs_test, y_pred):
    color = colores.get(pred, (0, 255, 255))
    cv2.rectangle(img_test_color, (x_min, y_min), (x_max, y_max), color, 2)

    label_text = f"{pred}"
    (w_text, h_text), _ = cv2.getTextSize(label_text, cv2.FONT_HERSHEY_SIMPLEX, 0.45, 1)
    cv2.rectangle(img_test_color, (x_min, y_min - h_text - 6), (x_min + w_text + 4, y_min), color, -1)
    cv2.putText(img_test_color, label_text, (x_min + 2, y_min - 4), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 255, 255), 1, cv2.LINE_AA)

cv2.imwrite("prediccion_test.jpg", img_test_color)

plt.figure(figsize=(10, 10))
plt.imshow(cv2.cvtColor(img_test_color, cv2.COLOR_BGR2RGB))
plt.title("Predicción Final sobre MPs_test.jpg")
plt.axis("off")
plt.show()

etiquetas_unicas = sorted(list(set(y_true + list(y_pred))))

print(f"\nAccuracy:  {accuracy_score(y_true, y_pred):.4f}")
print(f"Precision: {precision_score(y_true, y_pred, average='weighted', zero_division=0):.4f}")
print(f"Recall:    {recall_score(y_true, y_pred, average='weighted', zero_division=0):.4f}")
print(f"F1 Score:  {f1_score(y_true, y_pred, average='weighted', zero_division=0):.4f}\n")

conf_matrix = confusion_matrix(y_true, y_pred, labels=etiquetas_unicas)
plt.figure(figsize=(7, 5))
sns.heatmap(conf_matrix, annot=True, fmt='d', cbar=False, cmap='Blues', xticklabels=etiquetas_unicas, yticklabels=etiquetas_unicas)
plt.xlabel("Predicción", fontsize=12)
plt.ylabel("Real", fontsize=12)
plt.title("Matriz de Confusión", fontsize=13)
plt.tight_layout()
plt.savefig("matriz_confusion.png", dpi=300)
plt.show()